# TST Rate Constant Calculation

Converts NEB barriers + partial-Hessian vibrational frequencies into rate
constants k(T) using transition-state theory.

**Inputs** (from cluster runs):
- `neb_barrier.txt` files from Hop A and Hop B NEB jobs
- `vib_frequencies.json` files from IS + TS vibration jobs

**Outputs**:
- `rate_dict.json` — consumed by `kmc_calculation.ipynb`

---

**Physics**

| Quantity | Formula |
|---|---|
| Vineyard prefactor | ν = c · Π(ν_IS) / Π(ν_TS)  [s⁻¹] |
| ZPE correction | ΔE_zpe = Ea + ½·Σ(ν_TS − ν_IS)·h [eV] |
| Rate constant | k = ν · exp(−ΔE_zpe / kB T)  [s⁻¹] |


## Cell 1 — Imports & configuration

In [ ]:
import os, sys, json

parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from models.config import BASE_DIR

WORK_DIR    = os.path.join(BASE_DIR, 'calculation')
SUB_NEB_DIR = os.path.join(WORK_DIR, 'neb_subsurface')
VIB_DIR     = os.path.join(WORK_DIR, 'vibrations')
RESULTS_DIR = os.path.join(WORK_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Temperature ───────────────────────────────────────────────────────────────
T_K = 700.0    # K — edit for your target temperature

# ── Frequency filter ──────────────────────────────────────────────────────────
MIN_FREQ_CM1 = 50.0   # modes below this are excluded from Vineyard/ZPE

print(f'T_K        = {T_K} K')
print(f'RESULTS_DIR: {RESULTS_DIR}')

## Cell 2 — Load job metadata

Reads `hopa_jobs.json` and `hopb_jobs.json` written by the NEB orchestrators.

In [ ]:
with open(os.path.join(SUB_NEB_DIR, 'hopa', 'hopa_jobs.json')) as f:
    hopa_jobs = json.load(f)
with open(os.path.join(SUB_NEB_DIR, 'hopb', 'hopb_jobs.json')) as f:
    hopb_jobs = json.load(f)

print(f'Hop A jobs: {len(hopa_jobs)}')
print(f'Hop B jobs: {len(hopb_jobs)}')

## Cell 3 — Collect NEB barriers

In [ ]:
from models.tst_rates import collect_neb_results

neb_res_a = collect_neb_results(hopa_jobs, hop='hopa')
neb_res_b = collect_neb_results(hopb_jobs, hop='hopb')
neb_results = {**neb_res_a, **neb_res_b}

print(f'Loaded barriers: {len(neb_results)} labels')
for label, bd in list(neb_results.items())[:4]:
    print(f'  {label}: Ea={bd["E_abs"]:.3f} eV  Ed={bd["E_des"]:.3f} eV  converged={bd["converged"]}')

## Cell 4 — Split vib results into IS / TS dicts

Loads `vib_out` produced by `subsurface_neb_calculation.ipynb` Cell 6,
or reconstructs it from the files on disk.

In [ ]:
import glob
from models.tst_rates import split_vib_results

# Reconstruct vib_out from disk (label dirs inside VIB_DIR)
vib_out = {}
for label_dir in sorted(glob.glob(os.path.join(VIB_DIR, '*'))):
    label = os.path.basename(label_dir)
    vib_json = os.path.join(label_dir, 'vib_frequencies.json')
    vib_out[label] = {'vib_json': vib_json}

vib_is, vib_ts = split_vib_results(vib_out)

print(f'IS vib results: {len(vib_is)}')
print(f'TS vib results: {len(vib_ts)}')
missing = set(neb_results) - set(vib_is) - set(vib_ts)
if missing:
    print(f'WARNING: no vib data for {missing}')

## Cell 5 — Build rate dict at target T

In [ ]:
from models.tst_rates import build_rate_dict

rate_dict = build_rate_dict(
    neb_results   = neb_results,
    vib_results_is = vib_is,
    vib_results_ts = vib_ts,
    T_K           = T_K,
    apply_zpe     = True,
    min_freq_cm1  = MIN_FREQ_CM1,
)

print(f'Rate dict: {len(rate_dict)} labels')
for label, r in list(rate_dict.items())[:4]:
    print(f'  {label}: k_fwd={r["k_forward"]:.3e} s⁻¹  k_rev={r["k_reverse"]:.3e} s⁻¹  nu={r["nu"]:.3e} s⁻¹')

## Cell 6 — Sensitivity: k(T) vs T

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from models.tst_rates import arrhenius_rate

T_range = np.linspace(300, 1200, 200)

fig, ax = plt.subplots(figsize=(7, 4))
for label, r in rate_dict.items():
    nu  = r['nu']
    Ea  = r['Ea_zpe']
    k_T = [arrhenius_rate(nu, Ea, T) for T in T_range]
    ax.semilogy(T_range, k_T, lw=1.2, label=label, alpha=0.8)

ax.axvline(T_K, color='k', ls='--', lw=0.9, label=f'T = {T_K} K')
ax.set_xlabel('Temperature  [K]')
ax.set_ylabel('k  [s⁻¹]')
ax.set_title('Forward rate constants vs temperature')
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'rate_vs_T.png'), dpi=150)
plt.show()

## Cell 7 — Serialise to JSON

In [ ]:
from models.tst_rates import rates_to_json

out_path = os.path.join(RESULTS_DIR, f'rate_dict_T{int(T_K)}K.json')
rates_to_json(rate_dict, out_path)
print(f'Wrote: {out_path}')